# uDALES：读取东校区建筑几何

已提取四条道路围合区域内 146 条 OSM 建筑记录。高度含估算：9 条楼层数 × 3.3 m，137 条为 12 m 情景值。
本 Notebook 用 uDALES 官方读取器读取 STL，不启动模拟。Google Maps 核对和高度标定待完成。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').is_dir():
    ROOT = ROOT.parent
assert Path(sys.prefix).name.lower() == 'turbpy', '请选择 turbpy 内核'
sys.path[:0] = [str(ROOT / 'vendor/python-deps'), str(ROOT / 'vendor/u-dales/tools/python')]
from udgeom import UDGeom
case = ROOT / 'cases/001'
report = json.loads((case / 'geometry_report.json').read_text(encoding='utf-8'))
print('Python:', sys.executable)
print('Buildings:', report['building_records'])
print('Height sources:', report['height_sources'])


In [ ]:
buildings = UDGeom(case)
buildings.load('buildings.stl')
assert buildings.stl.is_watertight and buildings.stl.is_winding_consistent
qa = buildings.check(require_single_component=False)
assert qa['valid'], qa['issues']
print(qa['summary'])
surface = UDGeom(case)
surface.load('geom.001.stl')
surface_qa = surface.check(require_single_component=False)
assert surface_qa['valid'], surface_qa['issues']
print(surface_qa['summary'])


In [ ]:
from IPython.display import display, Image
display(Image(filename=str(case / 'geometry_overview.png')))


In [ ]:
status = json.loads((case / 'input_status.json').read_text(encoding='utf-8'))
print(json.dumps(status, ensure_ascii=False, indent=2))


`ibm_completed=false` 表示完整网格浸入边界输入尚未生成。实际执行 IBM 后报缺少 `ibm_preproc_f2py` 编译扩展。
`namoptions.001` 是前处理草案：runtime=0、热力关闭，不是稳定边界层最终模拟配置。
流程及重跑说明见 `docs/udales_workflow.md`。本机/HPC 选择留到真正启动模拟前。